# CX Intelligence — churn risk pipeline
Jalankan dengan akun kantor. Clear all outputs sebelum menyimpan/share notebook ini.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 2. Ambil kode dari GitHub dan cari folder ml/ secara otomatis
import os, glob, subprocess
os.chdir('/content')
REPO = 'https://github.com/athonk2511-del/CX-Intelligence.git'
!rm -rf /content/cx
r = subprocess.run(['git', 'clone', '--depth', '1', REPO, '/content/cx'], capture_output=True, text=True)
print(r.stdout, r.stderr)
hits = glob.glob('/content/cx/**/ml/config.yaml', recursive=True)
if not hits:
    print('GAGAL: ml/config.yaml tidak ditemukan. Isi repo:')
    !find /content/cx -maxdepth 3 -not -path '*/.git*'
    raise SystemExit
ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(ROOT)
print('OK, kode ditemukan di', ROOT)
!pip -q install -r ml/requirements.txt

In [ ]:
# 3. Salin data dari Drive ke disk lokal (jauh lebih cepat dibaca) + config
DRIVE_ROOT = '/content/drive/MyDrive/CX Intelligence'
LOCAL = '/content/data'
FAST = True  # True = grid kecil (4 kombinasi). False = grid penuh 36 kombinasi
import yaml, time
print('Isi folder:', os.listdir(DRIVE_ROOT))
t = time.time()
!rm -rf {LOCAL} && mkdir -p {LOCAL}
for d in os.listdir(DRIVE_ROOT):
    src = os.path.join(DRIVE_ROOT, d)
    if os.path.isdir(src):
        !cp -r "{src}" {LOCAL}/
        print('disalin:', d, len(os.listdir(src)), 'file')
print('selesai salin dalam %.0f detik' % (time.time() - t))
cfg = yaml.safe_load(open('ml/config.yaml'))
cfg['paths']['drive_root'] = LOCAL
if FAST:
    cfg['model']['grid'] = {'max_depth': [3, 5], 'learning_rate': [0.06], 'min_child_weight': [5], 'reg_lambda': [1, 5]}
    cfg['model']['n_estimators'] = 800
yaml.safe_dump(cfg, open('ml/config.local.yaml', 'w'), sort_keys=False, allow_unicode=True)

In [ ]:
# 4. Test
!python -m pytest ml/tests -q

In [ ]:
# 5. Train + score (log muncul per file / per tahap)
!PYTHONUNBUFFERED=1 python -u -m ml.churn_pipeline all --config ml/config.local.yaml

In [ ]:
# 6. Salin hasil ke Drive supaya dashboard bisa membacanya
out = os.path.join(LOCAL, cfg['paths']['output_folder'])
dst = os.path.join(DRIVE_ROOT, cfg['paths']['output_folder'])
!mkdir -p "{dst}" && cp -r "{out}"/. "{dst}"/
print('hasil disalin ke', dst, os.listdir(dst))